In [ ]:
# To do:
# Regarder toutes les colonnes disponibles
# Supprimer celles qui sont construites à partir de la puissance
# Garder seulement les informations qu'on connaîtrait avant de connaître la puissance

import pandas as pd
import matplotlib.pyplot as plt

# =====================================================
# 1. CHARGEMENT DES DONNÉES
# =====================================================

# Chargement du dataset nettoyé issu de la partie Big Data
df = pd.read_csv("export_IA.csv")

# =====================================================
# 2. CRÉATION DE LA CIBLE
# =====================================================

# Transformation de la puissance numérique en catégorie pour faire une classification
def categorie_puissance(p):

    if p <= 22:
        return "Normale"

    elif p <= 50:
        return "Rapide"

    elif p <= 150:
        return "Très rapide"

    else:
        return "Ultra rapide"

# Application de la fonction sur toute la colonne puissance_nominale
df["categorie_puissance"] = (
    df["puissance_nominale"]
    .apply(categorie_puissance)
)

# =====================================================
# 3. ANALYSE DE LA RÉPARTITION
# =====================================================

# Compter combien de bornes appartiennent à chaque catégorie
print("\nRépartition des catégories :\n")
print(df["categorie_puissance"].value_counts())

# Afficher la répartition en pourcentage
print("\nPourcentage :\n")
print(round(df["categorie_puissance"].value_counts(normalize=True) * 100, 2))

# Choix des variables utiles pour prédire la catégorie de puissance
features = [
    "implantation_station",
    "nbre_pdc",
    "prise_type_ef",
    "prise_type_2",
    "prise_type_combo_ccs",
    "prise_type_chademo",
    "prise_type_autre",
    "gratuit",
    "consolidated_latitude",
    "consolidated_longitude"
]

# On garde les variables explicatives + la cible
colonnes_utiles = features + ["categorie_puissance"]

# Suppression des lignes où au moins une valeur utile est manquante
df = df[colonnes_utiles].dropna()

print("\nTaille du dataset :", len(df))
print("\nColonnes retenues :")
print(df.columns)

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import joblib

# =====================================================
# 4. ENCODAGE DES VARIABLES
# =====================================================

# Correction des problèmes d'accents pour éviter des noms de colonnes illisibles
df["implantation_station"] = (
    df["implantation_station"]
    .str.normalize("NFKD")
    .str.encode("ascii", errors="ignore")
    .str.decode("utf-8")
)

# Transformation de la variable texte implantation_station en variables numériques
X = pd.get_dummies(df[features], columns=["implantation_station"])

# Nettoyage des booléens écrits sous forme de texte car les modèles ont besoin de nombres
X = X.replace({
    "true": 1,
    "false": 0,
    "True": 1,
    "False": 0,
    True: 1,
    False: 0
})

# Conversion forcée de toutes les colonnes en numérique
for col in X.columns:
    X[col] = pd.to_numeric(X[col], errors="coerce")

# Suppression des lignes qui auraient encore des valeurs invalides
X = X.dropna()

# On aligne df avec les lignes conservées dans X
df = df.loc[X.index]

# Conversion finale des variables explicatives en nombres décimaux
X = X.astype(float)
joblib.dump(X.columns.tolist(), "colonnes_puissance.pkl")
print("\nNombre de variables après encodage :")
print(X.shape[1])

# =====================================================
# 5. ENCODAGE DE LA CIBLE
# =====================================================

# Transformation des classes texte en nombres
le = LabelEncoder()

y = le.fit_transform(
    df["categorie_puissance"]
)

# Sauvegarde du LabelEncoder car le script final devra retraduire les nombres en texte
joblib.dump(
    le,
    "label_encoder_puissance.pkl"
)

print("\nClasses de puissance :")
print(le.classes_)

# =====================================================
# 6. TRAIN / TEST
# =====================================================

# Séparation des données : 80 % apprentissage, 20 % test
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y  # Garder les mêmes proportions de classes dans train et test
)

print("\nTaille train :", len(X_train))
print("Taille test :", len(X_test))

print("\nNombre de variables après encodage :")
print(X.shape[1])

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    classification_report
)

# =====================================================
# 7. RÉGRESSION LOGISTIQUE
# =====================================================

# Création d'un modèle simple utilisé comme modèle de comparaison
lr = LogisticRegression(
    max_iter=1000,
    random_state=42
)

# Entraînement du modèle sur les données d'apprentissage
lr.fit(X_train, y_train)

# Prédiction des catégories de puissance sur les données de test
y_pred_lr = lr.predict(X_test)

print("\n==========================")
print("RÉGRESSION LOGISTIQUE")
print("==========================")

# Accuracy : pourcentage global de bonnes prédictions
print("Accuracy :", accuracy_score(y_test, y_pred_lr))

# Rapport détaillé : précision, rappel et F1-score par classe
print(
    classification_report(
        y_test,
        y_pred_lr,
        target_names=le.classes_
    )
)

# =====================================================
# 8. RANDOM FOREST
# =====================================================

# Création d'un modèle Random Forest plus adapté aux relations complexes
rf = RandomForestClassifier(
    random_state=42
)

# Entraînement du Random Forest
rf.fit(X_train, y_train)

# Prédiction sur les données de test
y_pred_rf = rf.predict(X_test)

print("\n==========================")
print("RANDOM FOREST")
print("==========================")

# Affichage de l'accuracy du Random Forest
print(
    "Accuracy :",
    accuracy_score(y_test, y_pred_rf)
)

# Rapport détaillé du Random Forest
print(
    classification_report(
        y_test,
        y_pred_rf,
        target_names=le.classes_
    )
)

from sklearn.model_selection import GridSearchCV

# =====================================================
# 9. OPTIMISATION DU RANDOM FOREST AVEC GRIDSEARCHCV
# =====================================================

# Définition des hyperparamètres à tester
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5]
}

# GridSearchCV teste toutes les combinaisons d'hyperparamètres
grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=3,
    scoring="accuracy",
    n_jobs=-1,
    verbose=1
)

# Lancement de la recherche du meilleur modèle
grid_search.fit(X_train, y_train)

print("\n==========================")
print("GRIDSEARCHCV - RANDOM FOREST")
print("==========================")

# Affichage des meilleurs paramètres trouvés
print("Meilleurs paramètres :", grid_search.best_params_)

# Score moyen obtenu en validation croisée
print("Meilleur score CV :", grid_search.best_score_)

# Récupération du meilleur modèle trouvé
best_model = grid_search.best_estimator_

# Évaluation du meilleur modèle sur les données de test
y_pred_best = best_model.predict(X_test)

print("\nAccuracy du meilleur modèle :")
print(accuracy_score(y_test, y_pred_best))

# Rapport détaillé du meilleur modèle
print(classification_report(y_test, y_pred_best, target_names=le.classes_))

import joblib

# Sauvegarde du meilleur modèle pour le script final et la partie Web
joblib.dump(
    best_model,
    "modele_puissance.pkl"
)

# Sauvegarde du LabelEncoder pour traduire les prédictions numériques en texte
joblib.dump(
    le,
    "label_encoder_puissance.pkl"
)

print("Modèle sauvegardé.")

from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay)

# =====================================================
# 10. MATRICE DE CONFUSION
# =====================================================

# Création de la matrice de confusion : vrai label vs label prédit
cm = confusion_matrix(
    y_test,
    y_pred_best
)

# Préparation de l'affichage avec les noms des classes
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=le.classes_
)

fig, ax = plt.subplots(figsize=(8, 6))

# Affichage de la matrice
disp.plot(ax=ax)

plt.title("Matrice de confusion - Catégorie de puissance")

plt.tight_layout()

# Sauvegarde de la matrice sous forme d'image
plt.savefig("confusion_matrix_puissance.png", dpi=150)

plt.close()

import numpy as np

# =====================================================
# 11. IMPORTANCE DES VARIABLES
# =====================================================

# Récupération de l'importance calculée par le Random Forest
importances = best_model.feature_importances_

# Création d'un tableau avec le nom des variables et leur importance
importance_df = pd.DataFrame({
    "Variable": X.columns,
    "Importance": importances
})

# Tri des variables par importance croissante pour un affichage horizontal lisible
importance_df = importance_df.sort_values(
    by="Importance",
    ascending=True
)

plt.figure(figsize=(12, 8))

# Graphique horizontal des importances
plt.barh(
    importance_df["Variable"],
    importance_df["Importance"]
)

plt.xlabel("Importance")

plt.title(
    "Importance des variables pour la prédiction de la catégorie de puissance"
)

plt.tight_layout()

# Sauvegarde du graphique d'importance des variables
plt.savefig(
    "feature_importance_puissance.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()